In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

In [2]:
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

# **RAG IMPLEMENTATION WITH PDF DATA**

### STEP 1: Preparing document for text

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/Azure_essentials.pdf"

loader = PyPDFLoader(pdf_path)
#Create a document
docs = loader.load()

/var/folders/j9/fgc6h41s71n0gqpzy_jtd5hw0000gn/T/ipykernel_15319/2099187044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### STEP 2: Splitting documents into chunks

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=100,
)

chunks = splitter.split_documents(docs)

In [8]:
len(chunks)

546

## Creating own Metadata for PDF chunks

In [15]:
#PyPDF creates metadata for us
chunks[0].metadata

{'source': 'azure-essentials.pdf', 'developer': 'Microsoft'}

In [ ]:
for i in docs:
    i.metadata = {
        "source" : "azure-essentials.pdf",
        "developer" : "Microsoft"
    }

#Ran a loop in docs beacuse the list of docs is smaller than chunks

### STEP 3: Creating Embeddings for the chuncks

In [10]:
from langchain_openai import OpenAIEmbeddings

#the default is text-embedding-3-small
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

### STEP 4: Creating and Store Embedding in Vector Store

In [17]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

## Talk to LLM

In [18]:
# k similarity
context = vectorstore.similarity_search("Why do we need Azure? Tell me in one line", k=3)

In [19]:
context

[Document(metadata={'developer': 'Microsoft', 'source': 'azure-essentials.pdf'}, page_content='6 of 540 C H A P T E R 1  |  Getting started with Microsoft Azure \n \nanother eight regions. Only the largest global \nenterprises are able to deploy datacenters in this \nmanner, so using Azure makes it easy for \nenterprises of any size to deploy their services \nclose to their customers, wherever they are in the \nworld. And you can do that without ever leaving \nyour office. \nFor startups, Azure allows you to start with very \nlow cost and scale rapidly as you gain customers. \nYou would not face a large up-front capital \ninvestment to create a new VM—or even several \nnew VMs. The use of cloud computing fits well \nwith the scale fast, fail fast model of startup \ngrowth. \nAzure provides the flexibility to set up \ndevelopment and test configurations quickly. \nThese deployments can be scripted, giving you \nthe ability to spin up a development or test \nenvironment, do the testing, 

In [20]:
response = llm_openai.invoke(f"Why do we need Azure? Tell me in one line. you can answer using the following context: {context} ")
response.content

'We need Azure because it provides on-demand, globally distributed compute, storage, networking, and application services so organizations can develop, test, and scale applications quickly and cost‑effectively without owning datacenter hardware.'

this is an **in memory** vectorstore, this led us to rerun the notebook every time

We can persist the vectorstore in locally